In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ml.config import RANDOM_STATE
from ml.data import load_data, split_data
from ml.features import SmokerObeseFeature

In [ ]:
df = load_data()

X_train, X_test, y_train, y_test = split_data(df)
X_train.shape, X_test.shape

((1069, 6), (268, 6))

In [ ]:
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)
y_pred = dummy.predict(X_test)

y_pred[:5]

array([13030.20336929, 13030.20336929, 13030.20336929, 13030.20336929,
       13030.20336929])

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

mae, rmse, r2

(9861.797880026175, 13612.426003009037, -0.008391982350864469)

## Baseline

The `DummyRegressor` ignores all features and always predicts the **mean of the training target** (≈ 13,030) for every person.

| Metric | Value  | 
|--------|--------|
| MAE    | 9,862  |
| RMSE   | 13,612 |
| R²     | -0.008 |

- **MAE ≈ 9.9k**: guessing the mean, the error is about the size of a typical plan (median ≈ 9.4k).
- **RMSE is much higher than MAE** because a few huge errors dominate: smokers paying 40k+ still get the 13k guess.
- **R² ≈ 0** is expected: this model *is* "predict the mean". It is slightly negative because the training mean differs a bit from the test mean.

**This is the bar.** Any real model must reach an MAE below 9,862 and an R² above 0.


## Linear Regression

`LinearRegression` only works with numbers, so the data goes through a preprocessing step first:

- **Categorical** (`sex`, `smoker`, `region`) → `OneHotEncoder(drop="first")`: each category becomes a 0/1 column, dropping one redundant column per feature.
- **Numerical** (`age`, `bmi`, `children`) → `StandardScaler`: puts all columns on the same scale (mean 0, std 1).

Both steps are fitted **only on the training set** and wrapped with the model in a `Pipeline`, so the exact same transformation is applied to the test set and to new data (no leakage).


In [ ]:
numerical_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(drop="first"), categorical_features),
    ]
)

preprocessor.fit(X_train)
preprocessor.get_feature_names_out()

array(['num__age', 'num__bmi', 'num__children', 'cat__sex_male',
       'cat__smoker_yes', 'cat__region_northwest',
       'cat__region_southeast', 'cat__region_southwest'], dtype=object)

In [ ]:
preprocessor.transform(X_test.head(1)).shape

(1, 8)

In [ ]:
lr_pipe = Pipeline([("preprocessor", preprocessor), ("regressor", LinearRegression())])

lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)

In [ ]:
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = root_mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

mae_lr, rmse_lr, r2_lr

(4177.045561036326, 5956.342894363587, 0.8069287081198012)

In [ ]:
result = X_test.copy()

result["real"] = y_test
result["predicted"] = y_pred_lr
result["residual"] = result["real"] - result["predicted"]
result["obese"] = result["bmi"] >= 30

result.groupby(["smoker", "obese"])["residual"].mean()

smoker  obese
no      False    1371.561719
        True    -1750.799195
yes     False   -7463.443177
        True     9261.876105
Name: residual, dtype: float64

### Linear Regression results

| Metric | Dummy | Linear |
|---|---|---|
| MAE | 9,862 | 4,177 |
| RMSE | 13,612 | 5,956 |
| R² | -0.008 | 0.807 |

The linear model beats the baseline by far: the MAE drops by more than half and it explains ~81% of the variance.

**Where it fails** (mean residual = real − predicted, by group):
- **Non-smokers, not obese:** ≈ 1,371.56 **below** the real value (underestimates)
- **Non-smokers, obese:** ≈ 1,750.80 **above** the real value (overestimates)
- **Smokers, not obese:** ≈ 7,463.44 **above** the real value (overestimates)
- **Smokers, obese:** ≈ 9,261.88 **below** the real value (underestimates)

**Why:** the model learns a single weight for `smoker`, the same for every smoker. That weight ends up being an average of the two smoker groups: too small for obese smokers and too large for non-obese ones. It cannot represent the **smoker × BMI interaction** found in the EDA.


In [ ]:
coefficients = pd.Series(
    lr_pipe.named_steps["regressor"].coef_,
    index=lr_pipe.named_steps["preprocessor"].get_feature_names_out(),
)

coefficients.sort_values()

cat__region_southeast     -838.919616
cat__region_southwest     -659.139752
cat__region_northwest     -391.761455
cat__sex_male             -101.542054
num__children              636.501185
num__bmi                  1927.828251
num__age                  3472.975553
cat__smoker_yes          23077.764593
dtype: float64

### Coefficients

- **`smoker_yes` (+23k)** is by far the strongest factor: a smoker pays ~23k more than a non-smoker, all else equal.
- **`age` (+3.5k)** and **`bmi` (+1.9k)** come next. Since they were scaled, each value means "per 1 standard deviation" (~14 years of age, ~6 BMI points), which makes them comparable.
- **`sex`, `region` and `children`** barely change the price (hundreds).

The `smoker` weight is the same for every smoker, which is exactly why the model underestimates obese smokers and overestimates non-obese ones.


## Random Forest 

A Random Forest is a set of decision trees. Each tree splits the data with yes/no questions like "`bmi >= 30`?".

**No scaler needed:** a tree only compares a feature with a threshold, so it only cares about the order of the values, not their size. Scaling changes the numbers but keeps the order, so every split sends the same people to the same side.

**No `drop="first"`:** the linear model needed it because a redundant column makes the weights ambiguous. A tree has no weights, and keeping every category lets it isolate any region with a single question.

In [ ]:
rf_preprocessor = ColumnTransformer(
    [
        ("num", "passthrough", numerical_features),
        ("cat", OneHotEncoder(), categorical_features),
    ]
)

rf_preprocessor.fit(X_train)
rf_preprocessor.get_feature_names_out()

array(['num__age', 'num__bmi', 'num__children', 'cat__sex_female',
       'cat__sex_male', 'cat__smoker_no', 'cat__smoker_yes',
       'cat__region_northeast', 'cat__region_northwest',
       'cat__region_southeast', 'cat__region_southwest'], dtype=object)

In [ ]:
rf_pipe = Pipeline(
    [
        ("preprocessor", rf_preprocessor),
        ("regressor", RandomForestRegressor(random_state=RANDOM_STATE)),
    ]
)

rf_pipe.fit(X_train, y_train)
y_pred_rf = rf_pipe.predict(X_test)
y_pred_rf[:5]

array([10198.8425148,  6591.1503137, 11564.1999727, 42089.2778327,
        6030.794738 ])

In [ ]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = root_mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

mae_rf, rmse_rf, r2_rf

(2558.254771226865, 4642.16355496707, 0.882726686689128)

In [ ]:
result["predicted_rf"] = y_pred_rf
result["residual_rf"] = result["real"] - result["predicted_rf"]

result.groupby(["smoker", "obese"])[["residual", "residual_rf"]].mean()

residual  residual_rf
smoker obese                          
no     False  1371.561719  -317.166193
       True  -1750.799195  -733.054152
yes    False -7463.443177   851.077440
       True   9261.876105   805.545933

In [ ]:
result["abs_residual_rf"] = result["residual_rf"].abs()

result.groupby(["smoker", "obese"])["abs_residual_rf"].agg(["count", "mean", "max"])

count         mean           max
smoker obese                                  
no     False     93  2416.525255  20592.242533
       True     115  2714.524599  17629.411153
yes    False     22  2241.365009  15599.853811
       True      38  2615.659758  16559.237535

In [ ]:
y_pred_rf_train = rf_pipe.predict(X_train)
mae_rf_train = mean_absolute_error(y_train, y_pred_rf_train)
rmse_rf_train = root_mean_squared_error(y_train, y_pred_rf_train)
r2_rf_train = r2_score(y_train, y_pred_rf_train)

mae_rf_train, rmse_rf_train, r2_rf_train

(1040.2187743051452, 1870.3194106135363, 0.9744505486909902)

In [ ]:
importances = pd.Series(
    rf_pipe.named_steps["regressor"].feature_importances_,
    index=rf_pipe.named_steps["preprocessor"].get_feature_names_out(),
)

importances.sort_values()

cat__region_southwest    0.003680
cat__sex_male            0.004462
cat__sex_female          0.004486
cat__region_northeast    0.004787
cat__region_southeast    0.005354
cat__region_northwest    0.005892
num__children            0.022760
num__age                 0.135756
cat__smoker_yes          0.162315
num__bmi                 0.212045
cat__smoker_no           0.438462
dtype: float64

### Random Forest results

| Metric | Dummy | Linear | Random Forest |
|---|---|---|---|
| MAE | 9,862 | 4,177 | 2,558 |
| RMSE | 13,612 | 5,956 | 4,642 |
| R² | -0.008 | 0.807 | 0.883 |

The Random Forest is the best model on every metric.

**Smoker × BMI interaction solved** (mean residual = real − predicted, by group):

| Smoker / Obese | Linear | Random Forest |
|---|---|---|
| no / False | +1,372 | -317 |
| no / True | -1,751 | -733 |
| yes / False | -7,463 | +851 |
| yes / True | +9,262 | +806 |

Each tree can ask "smoker?" and then "`bmi >= 30`?", so it learns a different price for each group instead of a single average weight.

**Remaining errors:** the mean absolute error is similar in every group (~2.2k–2.7k), but every group still has a few errors of 15k–20k. These look like individual cases the 6 features cannot explain (noise), not a group the model ignores.

**Overfitting:** on the training set the RF reaches R² 0.974 (MAE 1,040) vs. 0.883 (MAE 2,558) on the test set. Fully grown trees memorize part of the noise. Still, the test score is what matters, and it is the best one so far.

**Feature importances:** `smoker` dominates (≈ 0.60, adding `smoker_no` and `smoker_yes`), followed by `bmi` (0.21) and `age` (0.14). Unlike the linear model, `bmi` beats `age`: the trees use it to split smokers into ~20k and ~40k, which the linear model could only express as one average weight.

## Linear Regression + smoker × obese

In [ ]:
int_preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(drop="first"), categorical_features),
        ("int", "passthrough", ["smoker_obese"]),
    ]
)

lr_int_pipe = Pipeline(
    [
        ("features", SmokerObeseFeature()),
        ("preprocessor", int_preprocessor),
        ("regressor", LinearRegression()),
    ]
)

In [ ]:
lr_int_pipe.fit(X_train, y_train)
y_pred_int = lr_int_pipe.predict(X_test)

y_pred_int[:5]

array([10195.48396077,  6757.61746956, 13403.00870653, 40392.56717625,
        6863.85036484])

In [ ]:
mae_int = mean_absolute_error(y_test, y_pred_int)
rmse_int = root_mean_squared_error(y_test, y_pred_int)
r2_int = r2_score(y_test, y_pred_int)

mae_int, rmse_int, r2_int

(2391.5618180969936, 4163.4102067636995, 0.9056685032940889)

In [ ]:
result["predicted_int"] = y_pred_int
result["residual_int"] = result["real"] - result["predicted_int"]


result.groupby(["smoker", "obese"])[["residual", "residual_int", "residual_rf"]].mean()

residual  residual_int  residual_rf
smoker obese                                        
no     False  1371.561719    -41.735857  -317.166193
       True  -1750.799195   -524.426517  -733.054152
yes    False -7463.443177    881.101894   851.077440
       True   9261.876105   1038.524681   805.545933

In [ ]:
int_coefficients = pd.Series(
    lr_int_pipe.named_steps["regressor"].coef_,
    index=lr_int_pipe.named_steps["preprocessor"].get_feature_names_out(),
)

int_coefficients.sort_values()

cat__region_southwest    -1118.341598
cat__region_southeast     -562.392826
cat__sex_male             -476.388409
cat__region_northwest     -144.160307
num__bmi                   296.347513
num__children              667.707980
num__age                  3626.473440
cat__smoker_yes          13292.365111
int__smoker_obese        19454.940734
dtype: float64

### Linear Regression + smoker × obese results

| Metric | Linear | Random Forest | Linear + smoker_obese |
|---|---|---|---|
| MAE | 4,177 | 2,558 | **2,392** |
| RMSE | 5,956 | 4,642 | **4,163** |
| R² | 0.807 | 0.883 | **0.906** |

A single extra 0/1 column, `smoker_obese` (smoker **and** `bmi >= 30`), made the linear model beat the Random Forest on every metric. It is created by `SmokerObeseFeature` (`ml/features.py`) as the first pipeline step and passes through the preprocessor unscaled, so its coefficient is in money units.

**Group bias gone** (mean residual = real − predicted):

| Smoker / Obese | Linear | Linear + smoker_obese | Random Forest |
|---|---|---|---|
| no / False | +1,372 | -42 | -317 |
| no / True | -1,751 | -524 | -733 |
| yes / False | -7,463 | +881 | +851 |
| yes / True | +9,262 | +1,039 | +806 |

**Coefficients now match the EDA:**
- `smoker_yes` (+13.3k): extra paid by a **non-obese** smoker (EDA: ~20k vs ~7k).
- `smoker_yes` + `smoker_obese` (13.3k + 19.5k = **+32.7k**): extra paid by an **obese** smoker (EDA: ~40k vs ~7k).
- `bmi` dropped from +1.9k to +0.3k: most of the "BMI effect" seen by the plain linear model was the jump of obese smokers. For non-smokers, BMI barely changes the price.

## Cross-validation

In [ ]:
cross_values_int = cross_val_score(
    lr_int_pipe, X_train, y_train, cv=5, scoring="neg_mean_absolute_error"
)
cross_values_rf = cross_val_score(
    rf_pipe, X_train, y_train, cv=5, scoring="neg_mean_absolute_error"
)

cross_values_int, cross_values_rf

(array([-2898.8261586 , -2397.16466377, -2495.81906398, -2380.08490671,
        -2538.89861073]),
 array([-2785.01587234, -2806.89697108, -2507.62927006, -2636.89211845,
        -3115.16191254]))

In [ ]:
pd.DataFrame(
    {
        "linear_int": [-cross_values_int.mean(), cross_values_int.std()],
        "random_forest": [-cross_values_rf.mean(), cross_values_rf.std()],
    },
    index=["mae_mean", "mae_std"],
)

,linear_int,random_forest
mean_mean,2542.158681,2770.319229
mean_std,187.968535,203.607076


### Cross-validation results

The comparison above uses a single test split (268 people), so part of the difference could be luck. 5-fold cross-validation on the **training set only** (the test set stays untouched):

| | Linear + smoker_obese | Random Forest |
|---|---|---|
| MAE (mean) | **2,542** | 2,770 |
| MAE (std) | 188 | 204 |

The linear model has the lower average error and wins in 4 of 5 folds, so its advantage is consistent, not a lucky split.

**Chosen model: Linear Regression + `smoker_obese`.** It is more accurate, simpler and explainable (each coefficient has a clear meaning). With only 9 weights it has little room to memorize noise, unlike the fully grown trees of the Random Forest (train R² 0.974 vs. test 0.883).